# 3단계: 택시 수요 급증 확인

**목적:** B035 택시운행분석 데이터에서 막차 이후(23:30~02:00) 택시 승차 건수가
폭발적으로 증가하는 구역을 확인하고, B009 KT 심야 유동인구와 교차 검증

**활용 데이터 (빅데이터캠퍼스 방문 후 수집):**
- B035 택시운행분석 (링크 단위 승하차 집계) — 빅데이터캠퍼스
- B009 KT 50m 월별 유동인구 (wlk_자치구_YYYYMM.txt) — 빅데이터캠퍼스

**출처:** 서울시 빅데이터 캠퍼스, B035 택시운행분석 / B009 서울시 50m 격자 KT 유동인구

**산출물:** 시간대별 수요 변화 그래프(PNG) + 수요 등치도(PNG) + 야간수요지수(CSV)

**반출 주의사항:**
- 소스코드 내 데이터 샘플 출력(head() 등) 제거 완료
- 지도 시각화는 geopandas PNG로 저장 (HTML 반출 불가)
- CSV 반출 시 반출신청서에 출처 및 산출과정 명시 필요

**B035 데이터 컬럼:**
- `T_LINK_ID`: 링크ID, `DAY`: 요일(1=월~7=일), `TIME`: 시간(0~23)
- `WEATHER`: 날씨, `DEST`: 목적지 링크ID
- `CNT_ON`: 승차건수, `CNT_OFF`: 하차건수, `CNT_EMP`: 공차건수

**B009 데이터 컬럼 (wlk 파일):**
- `셀id`, `x좌표`, `y좌표`: 격자 위치 (EPSG:5186)
- `요일`: 요일, `시간대`: 0~23
- `합계`: 유동인구 합계, `행정동코드`: 행정동코드, `기준년월`

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = '../data/'
OUTPUT_DIR = '../output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('라이브러리 로드 완료')

## 1. B035 택시운행분석 데이터 로드 (빅데이터캠퍼스 수집 데이터)

In [ ]:
# ── B035 스트리밍 집계 (체크포인트 포함)
# 중간에 오류 발생해도 이어서 실행 가능
# 체크포인트 파일: output/chk_b035_hourly.json / output/chk_b035_link.json / output/chk_b035_done.json

import glob, json as _json, gc, os

RENAME_MAP = {
    # 실제 컬럼명이 다를 경우 추가
    # 예: '링크ID': 'T_LINK_ID', '요일': 'DAY', '시간': 'TIME', '승차건수': 'CNT_ON'
}
NEED_COLS = ['T_LINK_ID', 'DAY', 'TIME', 'CNT_ON']

CHK_HOURLY = OUTPUT_DIR + 'chk_b035_hourly.json'
CHK_LINK   = OUTPUT_DIR + 'chk_b035_link.json'
CHK_DONE   = OUTPUT_DIR + 'chk_b035_done.json'

def detect_format(filepath):
    """파일의 인코딩·구분자·실제 컬럼명 탐지"""
    for enc in ['cp949', 'utf-8', 'euc-kr']:
        for sep in ['\t', ',', '|']:
            try:
                header = pd.read_csv(filepath, encoding=enc, sep=sep, nrows=0)
                if len(header.columns) > 2:
                    return enc, sep, list(header.columns)
            except Exception:
                continue
    raise ValueError(f'포맷 탐지 실패: {filepath}')

def save_checkpoint(hourly, link, done_files):
    with open(CHK_HOURLY, 'w', encoding='utf-8') as f: _json.dump(hourly, f)
    with open(CHK_LINK,   'w', encoding='utf-8') as f: _json.dump(link, f)
    with open(CHK_DONE,   'w', encoding='utf-8') as f: _json.dump(done_files, f)

def load_checkpoint():
    if all(os.path.exists(p) for p in [CHK_HOURLY, CHK_LINK, CHK_DONE]):
        with open(CHK_HOURLY) as f: hourly = {int(k): v for k, v in _json.load(f).items()}
        with open(CHK_LINK)   as f: link   = _json.load(f)
        with open(CHK_DONE)   as f: done   = _json.load(f)
        print(f'[체크포인트 복원] 완료 파일 {len(done)}개 / hourly {len(hourly)}개 / link {len(link):,}개')
        return hourly, link, done
    return {}, {}, []

# ── 체크포인트 로드 (이전에 중단된 경우 이어서 처리)
hourly_agg, link_agg, done_files = load_checkpoint()
total_rows  = 0
failed_files = []

taxi_files = (
    sorted(glob.glob(DATA_DIR + 'TaxiMach_Link_Dataset_Full_*.txt')) or
    sorted(glob.glob(DATA_DIR + 'B035/TaxiMach_Link_Dataset_Full_*.txt')) or
    sorted(glob.glob(DATA_DIR + 'TaxiMach_Link_Dataset_Full_*.csv')) or
    sorted(glob.glob(DATA_DIR + 'B035/TaxiMach_Link_Dataset_Full_*.csv'))
)
if not taxi_files:
    raise FileNotFoundError('B035 파일 없음')

# 이미 처리된 파일 건너뜀
remaining = [f for f in taxi_files if os.path.basename(f) not in done_files]
print(f'B035 파일 전체 {len(taxi_files)}개 | 잔여 {len(remaining)}개')
if not remaining:
    print('모든 파일 이미 처리 완료 → 체크포인트에서 로드')

for i, f in enumerate(remaining):
    fname = os.path.basename(f)
    try:
        enc, sep, raw_cols = detect_format(f)
        col_map   = {v: k for k, v in RENAME_MAP.items()}
        actual_need = [RENAME_MAP.get(c, c) for c in NEED_COLS]
        file_cols   = [col_map.get(c, c) for c in actual_need]
        use = [c for c in file_cols if c in raw_cols]

        df = pd.read_csv(f, encoding=enc, sep=sep, usecols=use)
        if RENAME_MAP:
            df = df.rename(columns=RENAME_MAP)

        df['TIME']   = pd.to_numeric(df['TIME'],   errors='coerce')
        df['DAY']    = pd.to_numeric(df['DAY'],    errors='coerce')
        df['CNT_ON'] = pd.to_numeric(df['CNT_ON'], errors='coerce').fillna(0)
        df = df[df['DAY'].isin([1,2,3,4,5])].dropna(subset=['TIME','DAY'])

        for t, cnt in df[df['TIME'].isin([20,21,22,23,0,1,2,3])].groupby('TIME')['CNT_ON'].sum().items():
            hourly_agg[int(t)] = hourly_agg.get(int(t), 0) + float(cnt)

        df_late = df[df['TIME'].isin([23,0,1,2])].copy()
        df_late['T_LINK_ID'] = df_late['T_LINK_ID'].astype(str)
        for lid, cnt in df_late.groupby('T_LINK_ID')['CNT_ON'].sum().items():
            link_agg[str(lid)] = link_agg.get(str(lid), 0) + float(cnt)

        total_rows += len(df)
        done_files.append(fname)
        print(f'  [{i+1}/{len(remaining)}] {fname} 완료 ({len(df):,}행)')
        del df, df_late
        gc.collect()

        # 파일마다 체크포인트 저장
        save_checkpoint(hourly_agg, link_agg, done_files)

    except Exception as e:
        failed_files.append(fname)
        print(f'  [{i+1}/{len(remaining)}] {fname} ⚠️ 오류 (건너뜀): {e}')
        gc.collect()

hourly_demand = (pd.DataFrame(list(hourly_agg.items()), columns=['TIME','승차건수'])
                   .sort_values('TIME').reset_index(drop=True))
df_link_night = pd.DataFrame(list(link_agg.items()), columns=['T_LINK_ID','야간승차건수'])

print(f'\n완료: 총 {len(done_files)}개 파일 처리')
if failed_files:
    print(f'⚠️  오류 파일 {len(failed_files)}개: {failed_files}')
print(f'시간대: {len(hourly_demand)}개 / 야간 링크: {len(df_link_night):,}개')
print('※ 오류로 재실행 시 셀을 다시 실행하면 자동으로 이어서 처리합니다')


In [ ]:
# cell-3에서 스트리밍 집계 완료
# hourly_demand: 시간대별 승차건수 (차트용)
# df_link_night: 링크별 야간 승차건수 (공간분석용)
print(f'시간대 집계: {len(hourly_demand)}개 시간대')
print(f'야간 링크 집계: {len(df_link_night):,}개 링크')

## 2. 막차 전후 수요 변화 분석

In [ ]:
# ── 시간대별 승차 건수 차트 (hourly_demand는 cell-3에서 이미 집계됨)
time_order = {20:0, 21:1, 22:2, 23:3, 0:4, 1:5, 2:6, 3:7}
hourly_demand['순서'] = hourly_demand['TIME'].map(time_order)
hourly_demand = hourly_demand.dropna(subset=['순서']).sort_values('순서')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hourly_demand['순서'], hourly_demand['승차건수'],
        marker='o', linewidth=2, color='#2E75B6', label='택시 승차 건수')
ax.axvline(x=3.5, color='red', linestyle='--', linewidth=1.5, label='막차 시점 (23:30)')
ax.fill_between(hourly_demand['순서'], hourly_demand['승차건수'],
                where=(hourly_demand['순서'] >= 3.5),
                alpha=0.15, color='red', label='막차 이후 구간')
ax.set_xticks(range(8))
ax.set_xticklabels(['20시','21시','22시','23시','0시','1시','2시','3시'])
ax.set_title('시간대별 택시 승차 건수 변화 (평일, B035)', fontsize=14)
ax.set_ylabel('승차 건수 (전 링크 합산)')
ax.legend()
ax.text(0.01, -0.08, '출처: 서울시 빅데이터 캠퍼스, B035 택시운행분석',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '03_hourly_demand.png', dpi=150)
plt.show()
print('그래프 저장 완료 → output/03_hourly_demand.png')

## 3. 행정동별 막차 이후 수요 집계

**B035는 링크(도로 구간) 단위 데이터입니다.** 행정동 단위로 집계하려면 링크ID → 행정동 매핑이 필요합니다.

**방법:** 캠퍼스 내 도로 네트워크 Shapefile(링크 형상 포함)을 이용해 링크 중심점을 구하고,  
행정동 경계 Shapefile과 공간 조인하여 `link_dong_map.csv`를 생성합니다.

> ※ Shapefile이 아직 없는 경우, Section 4(B009 KT 유동인구)의 `행정동코드`를 대신 활용합니다.

In [ ]:
# ── 행정동 경계 Shapefile 로드
gdf_dong = gpd.read_file(DATA_DIR + 'seoul_dong_boundary.shp', encoding='cp949')
if gdf_dong.crs.to_epsg() != 4326:
    gdf_dong = gdf_dong.to_crs('EPSG:4326')
print(f'행정동 경계 로드 완료: {len(gdf_dong)}개 동')
print('컬럼:', gdf_dong.columns.tolist())

In [ ]:
# ── 링크ID → 행정동 매핑 (df_link_night는 cell-3에서 이미 집계됨)

USE_LINK_MAP = False

# [방법 A] 링크-행정동 매핑 CSV
try:
    df_link_map = pd.read_csv(DATA_DIR + 'link_dong_map.csv', encoding='cp949')
    df_link_map.columns = df_link_map.columns.str.strip()
    df_link_map['T_LINK_ID'] = df_link_map['T_LINK_ID'].astype(str)
    print('[방법 A] link_dong_map.csv 로드 완료:', df_link_map.shape)
    USE_LINK_MAP = True
except FileNotFoundError:
    pass

# [방법 B] LINK WGS84 Shapefile → 링크 중심점 → 공간 조인
if not USE_LINK_MAP:
    link_shp_candidates = (
        glob.glob(DATA_DIR + 'LINK*WGS84*/*.shp') +
        glob.glob(DATA_DIR + 'LINK_WGS84*/*.shp') +
        glob.glob(DATA_DIR + 'B035/LINK*WGS84*/*.shp') +
        glob.glob(DATA_DIR + 'link_wgs84*/*.shp') +
        glob.glob(DATA_DIR + 'road_network.shp')
    )
    if link_shp_candidates:
        link_shp_path = link_shp_candidates[0]
        print(f'[방법 B] 링크 Shapefile: {link_shp_path}')
        try:
            gdf_link = gpd.read_file(link_shp_path, encoding='cp949')
            if gdf_link.crs is None or gdf_link.crs.to_epsg() != 4326:
                gdf_link = gdf_link.to_crs('EPSG:4326')
            # 링크ID 컬럼 자동 탐색
            lid_col = next((c for c in gdf_link.columns
                            if 'LINK' in c.upper() and 'ID' in c.upper()), gdf_link.columns[0])
            gdf_link = gdf_link.rename(columns={lid_col: 'T_LINK_ID'})
            gdf_link_pt = gpd.GeoDataFrame(
                {'T_LINK_ID': gdf_link['T_LINK_ID'].astype(str),
                 'geometry': gdf_link.geometry.centroid}, crs='EPSG:4326')
            gdf_joined = gpd.sjoin(gdf_link_pt, gdf_dong[['행정동코드','geometry']],
                                   how='left', predicate='within')
            df_link_map = gdf_joined[['T_LINK_ID','행정동코드']].dropna()
            print(f'[방법 B] 공간 조인 완료: {len(df_link_map):,}개 링크')
            USE_LINK_MAP = True
            del gdf_link, gdf_link_pt, gdf_joined
        except Exception as e:
            print(f'[방법 B] 오류: {e}')
    else:
        print('[방법 B] LINK WGS84 없음 → B009 대체 모드')

# ── df_link_night + 행정동 매핑 → 행정동별 야간 수요
if USE_LINK_MAP:
    df_merged = df_link_night.merge(df_link_map, on='T_LINK_ID', how='left')
    df_dong_demand = df_merged.groupby('행정동코드')['야간승차건수'].sum().reset_index()
    print(f'행정동별 야간 수요 집계 완료: {len(df_dong_demand)}개 동')
    del df_merged
else:
    print('링크 매핑 없음 → B009 심야 유동인구로 대체 진행')

## 4. B009 KT 심야 유동인구와 상관관계 교차 검증

**B009 KT 50m 월별 유동인구 (wlk 파일) 컬럼:**
- `셀id`, `x좌표`, `y좌표`: 격자 위치 (EPSG:5186 GRS80 TM 중부원점)
- `요일`: 요일 코드, `시간대`: 0~23 (정수)
- `합계`: 해당 셀-요일-시간대 유동인구 합계
- `행정동코드`: 행정동코드 (직접 포함 — Spatial Join 불필요)
- `기준년월`: YYYYMM

> ※ 파일명: `wlk_자치구_YYYYMM.txt` (구별 분리 파일 → 전처리 시 전 자치구 concat)

In [ ]:
# ── B009 KT 유동인구 로드 (체크포인트 포함)
# 중간에 오류 발생해도 이어서 실행 가능
# 체크포인트: output/chk_b009_dong.json / output/chk_b009_done.json

import glob, json as _json, gc, os

B009_RENAME = {
    # 실제 컬럼명이 다를 경우 추가
    # 예: 'ADMI_CD': '행정동코드', 'TIME_CD': '시간대', 'PEOPLE': '합계'
}

CHK_B009_DONG = OUTPUT_DIR + 'chk_b009_dong.json'
CHK_B009_DONE = OUTPUT_DIR + 'chk_b009_done.json'

def load_wlk(filepath):
    """B009 wlk .txt — 헤더 탐지 후 usecols 3개만 읽기"""
    enc, sep, cols = 'cp949', '\t', None
    for e in ['cp949', 'utf-8', 'euc-kr']:
        for s in ['\t', ',', '|']:
            try:
                h = pd.read_csv(filepath, encoding=e, sep=s, nrows=0)
                if len(h.columns) > 2:
                    enc, sep, cols = e, s, list(h.columns)
                    break
            except Exception:
                continue
        if cols:
            break

    if B009_RENAME:
        cols = [B009_RENAME.get(c, c) for c in cols]

    time_col = next((c for c in cols if '시간' in c), None)
    pop_col  = next((c for c in cols if '합계' in c or 'POP' in c.upper()), None)
    dong_col = next((c for c in cols if '행정동' in c and '코드' in c), None)

    rev = {v: k for k, v in B009_RENAME.items()}
    use_cols = [rev.get(c, c) for c in [time_col, pop_col, dong_col] if c]

    # usecols로 3개만 읽기 → 나이대별 컬럼 전혀 안 읽음
    df = pd.read_csv(filepath, encoding=enc, sep=sep, usecols=use_cols, dtype=str)
    if B009_RENAME:
        df = df.rename(columns=B009_RENAME)
    df = df.rename(columns={time_col:'시간대', pop_col:'합계', dong_col:'행정동코드'})
    df['시간대'] = pd.to_numeric(df['시간대'], errors='coerce')
    df['합계']   = pd.to_numeric(df['합계'],   errors='coerce').fillna(0)
    return df[df['시간대'].isin([23, 0, 1, 2])]

def save_b009_checkpoint(dong, done):
    with open(CHK_B009_DONG, 'w', encoding='utf-8') as f: _json.dump(dong, f, ensure_ascii=False)
    with open(CHK_B009_DONE, 'w', encoding='utf-8') as f: _json.dump(done, f, ensure_ascii=False)

def load_b009_checkpoint():
    if all(os.path.exists(p) for p in [CHK_B009_DONG, CHK_B009_DONE]):
        with open(CHK_B009_DONG, encoding='utf-8') as f: dong = _json.load(f)
        with open(CHK_B009_DONE, encoding='utf-8') as f: done = _json.load(f)
        print(f'[체크포인트 복원] 완료 파일 {len(done)}개 / 행정동 {len(dong)}개')
        return dong, done
    return {}, []

# ── 체크포인트 로드 (이전에 중단된 경우 이어서 처리)
dong_agg, done_b009 = load_b009_checkpoint()
failed_b009 = []

wlk_files = list(dict.fromkeys(
    sorted(glob.glob(DATA_DIR + 'B009/**/*.txt', recursive=True)) +
    sorted(glob.glob(DATA_DIR + 'wlk_*.txt')) +
    sorted(glob.glob(DATA_DIR + 'B009/**/*.csv', recursive=True)) +
    sorted(glob.glob(DATA_DIR + 'wlk_*.csv'))
))
if not wlk_files:
    raise FileNotFoundError('B009 파일 없음 — data/B009/ 또는 data/ 폴더 확인')

remaining_b009 = [f for f in wlk_files if os.path.basename(f) not in done_b009]
print(f'B009 파일 전체 {len(wlk_files)}개 | 잔여 {len(remaining_b009)}개')
if not remaining_b009:
    print('모든 파일 이미 처리 완료 → 체크포인트에서 로드')

for i, f in enumerate(remaining_b009):
    fname = os.path.basename(f)
    try:
        _df = load_wlk(f)
        _df['행정동코드'] = _df['행정동코드'].astype(str).str.zfill(8)
        for cd, val in _df.groupby('행정동코드')['합계'].mean().items():
            dong_agg.setdefault(str(cd), []).append(float(val))
        done_b009.append(fname)
        print(f'  [{i+1}/{len(remaining_b009)}] {fname} → 심야 {len(_df):,}행')
        del _df
        gc.collect()
        # 파일마다 체크포인트 저장
        save_b009_checkpoint(dong_agg, done_b009)
    except Exception as e:
        failed_b009.append(fname)
        print(f'  [{i+1}/{len(remaining_b009)}] {fname} ⚠️ 오류 (건너뜀): {e}')
        gc.collect()

df_pop_dong = pd.DataFrame(
    [{'행정동코드': k, '심야유동인구': sum(v)/len(v)} for k, v in dong_agg.items()]
)
print(f'\n심야 유동인구 집계 완료: {len(df_pop_dong)}개 행정동')
if failed_b009:
    print(f'⚠️  오류 파일 {len(failed_b009)}개: {failed_b009}')
print('※ 오류로 재실행 시 셀을 다시 실행하면 자동으로 이어서 처리합니다')

# ── 교차검증
if USE_LINK_MAP:
    df_dong_demand['행정동코드'] = df_dong_demand['행정동코드'].astype(str).str.zfill(8)
    df_corr = df_dong_demand.merge(df_pop_dong, on='행정동코드', how='inner')
    print(f'교차검증 병합: {len(df_corr)}개 행정동')
    corr, pval = stats.pearsonr(df_corr['야간승차건수'], df_corr['심야유동인구'])
    print(f'피어슨 상관계수: r={corr:.3f}, p={pval:.4f}')
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df_corr['심야유동인구'], df_corr['야간승차건수'], alpha=0.5, color='#2E75B6')
    ax.set_xlabel('심야 유동인구 (B009 KT)')
    ax.set_ylabel('택시 야간 승차 건수 (B035)')
    ax.set_title(f'심야 유동인구 vs 택시 수요 (r={corr:.3f}, p={pval:.3f})')
    ax.text(0.01, -0.10, '출처: 서울시 빅데이터 캠퍼스, B035 / B009',
            transform=ax.transAxes, fontsize=8, color='gray')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + '03_correlation.png', dpi=150)
    plt.show()
    print('산점도 저장 완료')
else:
    print('※ 링크 매핑 없음 → B009 기반 지수로 진행')
    df_corr = df_pop_dong.copy()


## 5. 수요 등치도(Choropleth) 생성

> ✅ **반출 가능:** geopandas로 PNG 파일로 저장 (HTML 파일 생성 없음)
>
> - 링크-행정동 매핑 완료 시: B035 야간 승차건수 기반 행정동 등치도
> - 링크-행정동 매핑 미완료 시: B009 심야 유동인구 기반 행정동 등치도 (대체)

In [ ]:
# ── 행정동 등치도 (geopandas Choropleth — PNG 반출 가능)

if USE_LINK_MAP:
    # [B035 기반] 행정동별 야간 승차건수
    plot_col = '야간승차건수'
    plot_title = '행정동별 막차 이후 택시 수요 (B035 야간 승차건수)'
    source_text = '출처: 서울시 빅데이터 캠퍼스, B035 택시운행분석'
    gdf_plot = gdf_dong.merge(
        df_dong_demand[['행정동코드', '야간승차건수']],
        on='행정동코드', how='left'
    )
    gdf_plot[plot_col] = gdf_plot[plot_col].fillna(0)
else:
    # [B009 대체] 심야 유동인구
    plot_col = '심야유동인구'
    plot_title = '행정동별 심야 유동인구 분포 (B009 KT, 23~02시)'
    source_text = '출처: 서울시 빅데이터 캠퍼스, B009 KT 50m 격자 유동인구'
    df_pop_dong_str = df_pop_dong.copy()
    df_pop_dong_str['행정동코드'] = df_pop_dong_str['행정동코드'].astype(str).str.zfill(8)
    gdf_dong['행정동코드'] = gdf_dong['행정동코드'].astype(str).str.zfill(8)
    gdf_plot = gdf_dong.merge(df_pop_dong_str, on='행정동코드', how='left')
    gdf_plot[plot_col] = gdf_plot[plot_col].fillna(0)

# ── 지도 시각화 (PNG 저장 - 반출 가능)
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
gdf_plot.plot(
    column=plot_col,
    ax=ax,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': plot_col, 'shrink': 0.6},
    edgecolor='white',
    linewidth=0.3,
    missing_kwds={'color': 'lightgrey'}
)
ax.set_title(plot_title, fontsize=14, pad=12)
ax.axis('off')
ax.text(0.01, -0.02, source_text,
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '03_demand_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('수요 등치도 저장 완료 → output/03_demand_heatmap.png')

## 6. 야간 수요 지수 정규화 및 저장

**산출 기준:**
- 링크-행정동 매핑 완료 시: B035 `야간승차건수` Min-Max 정규화
- 링크-행정동 매핑 미완료 시: B009 `심야유동인구` Min-Max 정규화 (대체 지수)

**반출:** 행정동 단위 단순집계 → 반출 신청서에 "B035 응용집계 / B009 응용집계" 명시

In [ ]:
# ── Min-Max 정규화 (단순집계 스케일링 - 반출 가능)
scaler = MinMaxScaler()

if USE_LINK_MAP:
    # B035 기반: 행정동별 야간 승차건수 정규화
    df_dong_demand['야간수요_지수'] = scaler.fit_transform(
        df_dong_demand[['야간승차건수']].fillna(0)
    )
    # 행정동명 병합
    if '행정동명' in gdf_dong.columns:
        df_dong_demand = df_dong_demand.merge(
            gdf_dong[['행정동코드', '행정동명']], on='행정동코드', how='left'
        )
    # CSV 저장 (반출 가능: 행정동 단위 응용집계)
    df_out = df_dong_demand[['행정동코드', '야간승차건수', '야간수요_지수']].copy()
    if '행정동명' in df_dong_demand.columns:
        df_out.insert(1, '행정동명', df_dong_demand['행정동명'])
    df_out.to_csv(OUTPUT_DIR + 'stage03_taxi_demand.csv', index=False, encoding='utf-8-sig')
    print('3단계 결과 저장 완료 → output/stage03_taxi_demand.csv')
    print(f'총 {len(df_out)}개 행정동 야간 수요 지수 산출 (B035 기반)')

    # 상위 10개 구역 확인 (집계 건수만 출력)
    top10 = df_out.sort_values('야간수요_지수', ascending=False).head(10)
    if '행정동명' in top10.columns:
        print('\n[야간 택시 수요 상위 10개 행정동]')
        print(top10[['행정동명', '야간수요_지수']].to_string(index=False))
    else:
        print('\n[야간 택시 수요 상위 10개 행정동코드]')
        print(top10[['행정동코드', '야간수요_지수']].to_string(index=False))

else:
    # B009 대체: 심야 유동인구 기반 수요 지수
    df_pop_dong['야간수요_지수'] = scaler.fit_transform(
        df_pop_dong[['심야유동인구']].fillna(0)
    )
    if '행정동명' in gdf_dong.columns:
        gdf_dong['행정동코드'] = gdf_dong['행정동코드'].astype(str).str.zfill(8)
        df_pop_dong = df_pop_dong.merge(
            gdf_dong[['행정동코드', '행정동명']], on='행정동코드', how='left'
        )
    df_out = df_pop_dong[['행정동코드', '심야유동인구', '야간수요_지수']].copy()
    if '행정동명' in df_pop_dong.columns:
        df_out.insert(1, '행정동명', df_pop_dong['행정동명'])
    df_out.to_csv(OUTPUT_DIR + 'stage03_taxi_demand.csv', index=False, encoding='utf-8-sig')
    print('3단계 결과 저장 완료 → output/stage03_taxi_demand.csv (B009 대체 지수)')
    print(f'총 {len(df_out)}개 행정동 야간 수요 지수 산출')
    print('※ Shapefile 수령 후 B035 링크-행정동 매핑으로 재산출 권장')

    top10 = df_out.sort_values('야간수요_지수', ascending=False).head(10)
    if '행정동명' in top10.columns:
        print('\n[심야 유동인구 상위 10개 행정동]')
        print(top10[['행정동명', '야간수요_지수']].to_string(index=False))

## 🛠 트러블슈팅 & 수정 가이드

> 오류 발생 시 이 셀을 먼저 읽고, 아래 **진단 코드 셀**을 실행하세요.

---

### 1. 파일을 못 찾는 오류 (`FileNotFoundError`)

```
FileNotFoundError: B035 파일 없음
```

**원인:** 파일이 예상 폴더에 없거나 파일명 패턴이 다름

**수정 위치:** `cell-3` 상단의 `taxi_files` glob 패턴

```python
# 현재 코드 (이 중 하나가 맞아야 함)
DATA_DIR + 'TaxiMach_Link_Dataset_Full_*.txt'
DATA_DIR + 'B035/TaxiMach_Link_Dataset_Full_*.txt'

# 파일명이 다를 경우 예시 (실제 파일명에 맞게 수정)
DATA_DIR + 'B035/TAXI_LINK_*.txt'
DATA_DIR + 'B035/*.txt'   # 패턴을 넓게
```

---

### 2. 컬럼명을 못 찾는 오류 (`KeyError`, `usecols` 오류)

```
ValueError: Usecols do not match columns
KeyError: 'CNT_ON'
```

**원인:** 실제 파일의 컬럼명이 코드에서 기대하는 이름과 다름

**수정 위치:** `cell-3` 상단의 `RENAME_MAP`

```python
# ↓ 아래 진단 셀 실행 → 실제 컬럼명 확인 후 여기에 추가
RENAME_MAP = {
    '실제컬럼명': 'T_LINK_ID',   # 링크ID
    '실제컬럼명': 'DAY',          # 요일
    '실제컬럼명': 'TIME',         # 시간
    '실제컬럼명': 'CNT_ON',       # 승차건수
}
```

B009도 동일: `cell-11` 상단의 `B009_RENAME`

```python
B009_RENAME = {
    '실제컬럼명': '행정동코드',
    '실제컬럼명': '시간대',
    '실제컬럼명': '합계',
}
```

---

### 3. Shapefile 행정동코드 컬럼명 오류

```
KeyError: '행정동코드'
```

**수정 위치:** `cell-8` 실행 후 출력된 컬럼명 확인 → `cell-9`에서 수정

```python
# cell-9에서 '행정동코드' 대신 실제 컬럼명으로 바꾸기
# 예: 'ADM_CD', 'HDONG_CD', 'dong_code' 등
gdf_dong = gdf_dong.rename(columns={'실제컬럼명': '행정동코드'})
```

---

### 4. 인코딩 오류 (`UnicodeDecodeError`)

```
UnicodeDecodeError: 'cp949' codec can't decode
```

**수정 위치:** `detect_format()` 함수의 인코딩 리스트에 추가

```python
for enc in ['cp949', 'utf-8', 'euc-kr', 'latin-1']:  # latin-1 추가
```

---

### 5. 메모리 부족 (`MemoryError`, 커널 재시작)

**대처법:**
1. 다른 노트북 모두 닫기
2. 셀 다시 실행 → 체크포인트에서 자동으로 이어서 처리
3. 그래도 부족하면 `cell-3`에서 처리 월 범위 줄이기:

```python
# 예: 2015년만 처리
taxi_files = [f for f in taxi_files if '2015' in f]
```

---

### 6. LINK WGS84 Shapefile을 못 찾음

```
[방법 B] LINK WGS84 없음 → B009 대체 모드
```

**수정 위치:** `cell-9`의 `link_shp_candidates` 리스트에 실제 경로 추가

```python
link_shp_candidates = (
    glob.glob(DATA_DIR + 'LINK*WGS84*/*.shp') +
    glob.glob(DATA_DIR + '실제폴더명/*.shp')  # ← 실제 폴더명 추가
)
```

---

### 7. 체크포인트 초기화 (처음부터 다시 돌리고 싶을 때)

```python
import os
for f in ['chk_b035_hourly.json','chk_b035_link.json','chk_b035_done.json',
          'chk_b009_dong.json','chk_b009_done.json']:
    p = OUTPUT_DIR + f
    if os.path.exists(p): os.remove(p)
print('체크포인트 초기화 완료 — 처음부터 재실행됩니다')
```


In [ ]:
# ── 진단 스크립트
# 전체 실행 시 자동으로 돌지 않음
# 오류 발생 시: RUN_DIAGNOSTICS = True 로 바꾸고 이 셀만 실행

RUN_DIAGNOSTICS = False  # ← True 로 바꾸면 실행됨

if RUN_DIAGNOSTICS:
    # ── 진단 스크립트: 오류 발생 시 이 셀 먼저 실행
    # 파일 존재 여부, 실제 컬럼명, 인코딩, 행수를 자동으로 출력
    
    import glob, os
    
    print("=" * 60)
    print("진단 결과")
    print("=" * 60)
    
    # ── 1. 폴더 구조 확인
    print("\n[1] DATA_DIR 내 파일 목록")
    for root, dirs, files in os.walk(DATA_DIR):
        level = root.replace(DATA_DIR, "").count(os.sep)
        indent = "  " * level
        rel = os.path.relpath(root, DATA_DIR)
        print(f"  {indent}{rel}/")
        for fn in sorted(files)[:5]:
            size = os.path.getsize(os.path.join(root, fn)) / 1024**2
            print(f"  {indent}  {fn}  ({size:.1f} MB)")
        if len(files) > 5:
            print(f"  {indent}  ... 외 {len(files)-5}개")
    
    # ── 2. B035 파일 컬럼명 확인
    print("\n[2] B035 첫 번째 파일 컬럼명")
    b035_files = (
        sorted(glob.glob(DATA_DIR + "TaxiMach_Link_Dataset_Full_*.txt")) or
        sorted(glob.glob(DATA_DIR + "B035/TaxiMach_Link_Dataset_Full_*.txt")) or
        sorted(glob.glob(DATA_DIR + "B035/*.txt")) or
        sorted(glob.glob(DATA_DIR + "**/*.txt", recursive=True))
    )
    if b035_files:
        f = b035_files[0]
        print(f"  파일: {os.path.basename(f)}")
        for enc in ["cp949", "utf-8", "euc-kr"]:
            for sep in ["\t", ",", "|"]:
                try:
                    h = pd.read_csv(f, encoding=enc, sep=sep, nrows=3)
                    if len(h.columns) > 2:
                        print(f"  인코딩: {enc}  구분자: {repr(sep)}")
                        print(f"  컬럼({len(h.columns)}개): {list(h.columns)}")
                        expected = ["T_LINK_ID","DAY","TIME","CNT_ON"]
                        missing = [c for c in expected if c not in h.columns]
                        if missing:
                            print(f"  ⚠️  없는 컬럼: {missing} → RENAME_MAP 수정 필요")
                        else:
                            print("  ✅ 필요 컬럼 모두 확인됨")
                        break
                except Exception: continue
            else: continue
            break
    else:
        print("  ❌ B035 파일 없음 → 경로/파일명 확인 필요")
    
    # ── 3. B009 파일 컬럼명 확인
    print("\n[3] B009 첫 번째 파일 컬럼명")
    b009_files = (
        sorted(glob.glob(DATA_DIR + "B009/**/*.txt", recursive=True)) or
        sorted(glob.glob(DATA_DIR + "wlk_*.txt"))
    )
    if b009_files:
        f = b009_files[0]
        print(f"  파일: {os.path.basename(f)}")
        for enc in ["cp949", "utf-8", "euc-kr"]:
            for sep in ["\t", ",", "|"]:
                try:
                    h = pd.read_csv(f, encoding=enc, sep=sep, nrows=3)
                    if len(h.columns) > 2:
                        print(f"  인코딩: {enc}  구분자: {repr(sep)}")
                        print(f"  컬럼({len(h.columns)}개): {list(h.columns)}")
                        chk = {"시간대":False, "합계":False, "행정동코드":False}
                        for c in h.columns:
                            if "시간" in c: chk["시간대"] = True
                            if "합계" in c: chk["합계"] = True
                            if "행정동" in c and "코드" in c: chk["행정동코드"] = True
                        missing = [k for k, v in chk.items() if not v]
                        if missing:
                            print(f"  ⚠️  못 찾은 컬럼: {missing} → B009_RENAME 수정 필요")
                        else:
                            print("  ✅ 필요 컬럼 모두 확인됨")
                        break
                except Exception: continue
            else: continue
            break
    else:
        print("  ❌ B009 파일 없음 → 경로/파일명 확인 필요")
    
    # ── 4. Shapefile 확인
    print("\n[4] Shapefile 확인")
    dong_shp = DATA_DIR + "seoul_dong_boundary.shp"
    if os.path.exists(dong_shp):
        import geopandas as gpd
        gdf = gpd.read_file(dong_shp, encoding="cp949")
        print(f"  행정동 경계: ✅ {len(gdf)}개 동")
        print(f"  컬럼: {list(gdf.columns)}")
        dong_code_col = [c for c in gdf.columns if "동" in c or "code" in c.lower() or "CD" in c]
        print(f"  행정동코드 후보: {dong_code_col}")
    else:
        print("  ❌ seoul_dong_boundary.shp 없음")
    
    link_shps = (
        glob.glob(DATA_DIR + "LINK*WGS84*/*.shp") +
        glob.glob(DATA_DIR + "B035/LINK*WGS84*/*.shp") +
        glob.glob(DATA_DIR + "link_wgs84*/*.shp")
    )
    if link_shps:
        print(f"  LINK WGS84: ✅ {link_shps[0]}")
    else:
        print("  LINK WGS84: ❌ 없음 → B009 대체 모드로 진행")
    
    # ── 5. 체크포인트 상태
    print("\n[5] 체크포인트 상태")
    chk_files = {
        "B035 hourly": OUTPUT_DIR + "chk_b035_hourly.json",
        "B035 link":   OUTPUT_DIR + "chk_b035_link.json",
        "B035 done":   OUTPUT_DIR + "chk_b035_done.json",
        "B009 dong":   OUTPUT_DIR + "chk_b009_dong.json",
        "B009 done":   OUTPUT_DIR + "chk_b009_done.json",
    }
    import json as _json
    for name, path in chk_files.items():
        if os.path.exists(path):
            with open(path) as _f: data = _json.load(_f)
            print(f"  {name}: ✅ {len(data)}개 항목")
        else:
            print(f"  {name}: — 없음")
    
    print("\n" + "=" * 60)
    print("진단 완료 — 위 결과를 보고 가이드(위 마크다운 셀) 참조하여 수정하세요")
    print("=" * 60)
    


### 체크포인트 초기화 (필요 시만 실행)

In [ ]:
# ── 체크포인트 초기화
# 전체 실행 시 자동으로 돌지 않음
# 처음부터 다시 돌리고 싶을 때: RESET_CHECKPOINT = True 로 바꾸고 이 셀만 실행

RESET_CHECKPOINT = False  # ← True 로 바꾸면 실행됨 (주의: 진행상황 삭제)

if RESET_CHECKPOINT:
    # ── 체크포인트 초기화 (처음부터 다시 돌리고 싶을 때만 실행)
    # 주의: 실행하면 지금까지 처리된 결과가 모두 삭제됩니다
    
    import os
    CHK_FILES = [
        "chk_b035_hourly.json", "chk_b035_link.json", "chk_b035_done.json",
        "chk_b009_dong.json",   "chk_b009_done.json"
    ]
    removed = []
    for fn in CHK_FILES:
        p = OUTPUT_DIR + fn
        if os.path.exists(p):
            os.remove(p)
            removed.append(fn)
    if removed:
        print(f"삭제됨: {removed}")
        print("cell-3, cell-11 다시 실행하면 처음부터 재처리됩니다")
    else:
        print("체크포인트 파일 없음 (이미 초기화 상태)")
    
